# 🔍 SO1-A — Détection de Texte Manipulateur avec TrOCR
## Projet CBL — Media Credibility | Module : Manipulation des Médias

---

### 🎯 Objectif
Utiliser **TrOCR (Microsoft)** — le modèle gagnant de la comparaison SO1-A — pour extraire le texte des images publicitaires manipulatrices.

### 📋 Structure du notebook
1. **Exploration du dataset** — visualisations et exemples
2. **Modèle TrOCR** — architecture et inférence
3. **XAI** — 3 méthodes d'explicabilité
4. **Évaluation des performances** — métriques détaillées

### 🏆 Pourquoi TrOCR ?
TrOCR a obtenu **100% de taux de détection** (20/20 images) contre 20% pour CRNN et 95% pour Tesseract lors de la comparaison initiale. Son architecture Transformer lui permet de comprendre le contexte global de l'image, idéale pour les textes publicitaires stylisés.

---

## ⚙️ PARTIE 0 — Installation et Imports

In [ ]:
!pip install transformers torch torchvision Pillow opencv-python matplotlib seaborn tqdm captum -q
print('✅ Librairies installées !')

In [ ]:
import os, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
from PIL import Image
from tqdm import tqdm
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn.functional as F
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# ── Configuration ────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Chemins dataset ──────────────────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/jeffaudi/coco-2014-dataset-for-yolov3/coco2014'
IMAGES_TRAIN = os.path.join(DATASET_PATH, 'images', 'train2014')

print(f'✅ PyTorch : {torch.__version__}')
print(f'✅ Device  : {device}')
print(f'✅ Dataset : {IMAGES_TRAIN}')
print(f'✅ Existe  : {os.path.exists(IMAGES_TRAIN)}')

---
# 📊 PARTIE 1 — Exploration du Dataset

Avant tout entraînement, il est essentiel de **comprendre et visualiser les données**. Cette partie explore le dataset COCO 2014, analyse sa composition, et identifie les images contenant du texte visible — ce qui est notre cible pour l'OCR.

In [ ]:
# ============================================================
# 1.1 — Statistiques globales du dataset
# ============================================================
all_images = [f for f in os.listdir(IMAGES_TRAIN) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

# Analyser les tailles d'images
sizes = []
sample_for_stats = random.sample(all_images, min(500, len(all_images)))
for img_name in tqdm(sample_for_stats, desc='Analyse des dimensions'):
    try:
        img = Image.open(os.path.join(IMAGES_TRAIN, img_name))
        sizes.append(img.size)  # (largeur, hauteur)
    except:
        pass

widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]

print('='*55)
print('📊 STATISTIQUES DU DATASET COCO 2014')
print('='*55)
print(f'  Nombre total d\'images   : {len(all_images):,}')
print(f'  Largeur  — moy: {np.mean(widths):.0f}px | min: {min(widths)}px | max: {max(widths)}px')
print(f'  Hauteur  — moy: {np.mean(heights):.0f}px | min: {min(heights)}px | max: {max(heights)}px')
print(f'  Format   — 100% JPEG')
print('='*55)

In [ ]:
# ============================================================
# 1.2 — Visualisation des distributions
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('📊 Distribution des propriétés du dataset COCO 2014', fontsize=14, fontweight='bold')

# Distribution des largeurs
axes[0].hist(widths, bins=30, color='#3498db', edgecolor='white', linewidth=0.5)
axes[0].axvline(np.mean(widths), color='red', linestyle='--', linewidth=2, label=f'Moy: {np.mean(widths):.0f}px')
axes[0].set_title('Distribution des largeurs', fontweight='bold')
axes[0].set_xlabel('Largeur (pixels)')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# Distribution des hauteurs
axes[1].hist(heights, bins=30, color='#e74c3c', edgecolor='white', linewidth=0.5)
axes[1].axvline(np.mean(heights), color='blue', linestyle='--', linewidth=2, label=f'Moy: {np.mean(heights):.0f}px')
axes[1].set_title('Distribution des hauteurs', fontweight='bold')
axes[1].set_xlabel('Hauteur (pixels)')
axes[1].set_ylabel('Fréquence')
axes[1].legend()

# Scatter largeur vs hauteur
axes[2].scatter(widths[:200], heights[:200], alpha=0.4, color='#2ecc71', s=15)
axes[2].set_title('Largeur vs Hauteur', fontweight='bold')
axes[2].set_xlabel('Largeur (pixels)')
axes[2].set_ylabel('Hauteur (pixels)')
axes[2].plot([0, max(widths)], [0, max(widths)], 'r--', alpha=0.4, label='Carré (L=H)')
axes[2].legend()

plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Distribution sauvegardée')

In [ ]:
# ============================================================
# 1.3 — Aperçu visuel du dataset (grille d'exemples)
# ============================================================
random.seed(42)
sample_24 = random.sample(all_images, 24)

fig, axes = plt.subplots(4, 6, figsize=(22, 15))
fig.suptitle('📸 Aperçu du dataset COCO 2014 — 24 images aléatoires',
             fontsize=15, fontweight='bold', y=1.01)

for idx, (ax, img_name) in enumerate(zip(axes.flatten(), sample_24)):
    img = Image.open(os.path.join(IMAGES_TRAIN, img_name)).convert('RGB')
    w, h = img.size
    ax.imshow(img)
    ax.set_title(f'{w}×{h}', fontsize=8, color='#555')
    ax.axis('off')

plt.tight_layout()
plt.savefig('dataset_apercu.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Aperçu sauvegardé')

In [ ]:
# ============================================================
# 1.4 — Analyse des canaux de couleur (RGB)
# L'analyse des canaux permet de comprendre si les images
# sont naturellement colorées ou tendent vers le monochrome
# ============================================================
random.seed(42)
color_sample = random.sample(all_images, 100)

r_means, g_means, b_means = [], [], []
for img_name in color_sample:
    img = np.array(Image.open(os.path.join(IMAGES_TRAIN, img_name)).convert('RGB'))
    r_means.append(img[:, :, 0].mean())
    g_means.append(img[:, :, 1].mean())
    b_means.append(img[:, :, 2].mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('🎨 Analyse des canaux de couleur', fontsize=13, fontweight='bold')

# Histogramme des moyennes par canal
axes[0].hist(r_means, bins=20, color='red',   alpha=0.6, label='Rouge')
axes[0].hist(g_means, bins=20, color='green', alpha=0.6, label='Vert')
axes[0].hist(b_means, bins=20, color='blue',  alpha=0.6, label='Bleu')
axes[0].set_title('Distribution des moyennes RGB')
axes[0].set_xlabel('Valeur moyenne du canal (0-255)')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# Barres moyennes par canal
canal_means = [np.mean(r_means), np.mean(g_means), np.mean(b_means)]
bars = axes[1].bar(['Rouge', 'Vert', 'Bleu'], canal_means,
                    color=['red', 'green', 'blue'], edgecolor='black', alpha=0.7)
axes[1].set_title('Moyenne globale par canal RGB')
axes[1].set_ylabel('Valeur moyenne')
axes[1].set_ylim(0, 255)
for bar, val in zip(bars, canal_means):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{val:.1f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('analyse_couleurs.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Moyennes RGB : R={np.mean(r_means):.1f} | G={np.mean(g_means):.1f} | B={np.mean(b_means):.1f}')

In [ ]:
# ============================================================
# 1.5 — Sélection des images de test (avec texte visible)
# On sélectionne manuellement des images contenant du texte
# ============================================================
random.seed(42)
candidates = random.sample(all_images, 50)

# Afficher la grille pour sélection visuelle
fig, axes = plt.subplots(5, 10, figsize=(30, 15))
fig.suptitle('🔍 Sélection manuelle — cherche les images avec texte visible',
             fontsize=13, fontweight='bold')

for idx, (ax, img_name) in enumerate(zip(axes.flatten(), candidates)):
    img = Image.open(os.path.join(IMAGES_TRAIN, img_name)).convert('RGB')
    ax.imshow(img)
    ax.set_title(f'#{idx}', fontsize=7, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()
print('👆 Note les numéros des images avec du texte visible')

In [ ]:
# ============================================================
# 1.6 — Constitution du set de test final (20 images)
# ============================================================
TEST_IMAGES = candidates[:20]  # Prend les 20 premières

# Afficher les 20 images de test clairement
fig, axes = plt.subplots(4, 5, figsize=(20, 16))
fig.suptitle('🧪 20 Images de Test — Input de TrOCR',
             fontsize=14, fontweight='bold')

for idx, (ax, img_name) in enumerate(zip(axes.flatten(), TEST_IMAGES)):
    img_path = os.path.join(IMAGES_TRAIN, img_name)
    img = Image.open(img_path).convert('RGB')
    w, h = img.size
    ax.imshow(img)
    ax.set_title(f'Test #{idx+1}\n{w}×{h}px', fontsize=9, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('images_test.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ {len(TEST_IMAGES)} images sélectionnées pour le test TrOCR')

---
# 🤖 PARTIE 2 — Modèle TrOCR

## Architecture TrOCR — Comment ça marche ?

```
Image de texte (ex: pub avec "SOLDES -70%")
         ↓
┌─────────────────────────────────────────┐
│  ENCODER : ViT (Vision Transformer)     │
│  • Image découpée en patches 16×16      │
│  • Chaque patch → embedding 768-dim     │
│  • Self-Attention : chaque patch        │
│    regarde TOUS les autres patches      │
│  • Comprend le contexte visuel global   │
└──────────────────┬──────────────────────┘
                   ↓ cross-attention
┌─────────────────────────────────────────┐
│  DECODER : GPT-2 like                   │
│  • Génère le texte token par token      │
│  • "S" → "O" → "L" → "D" → "E" → "S"   │
│  • S'arrête au token EOS                │
└─────────────────────────────────────────┘
         ↓
"SOLDES -70%"  (texte extrait)
```

**Pourquoi TrOCR est meilleur que CRNN pour notre cas ?**
- CRNN lit le texte colonne par colonne (gauche → droite) : limité
- TrOCR regarde l'image **entière en une fois** via l'attention : capte mieux les textes stylisés, obliques, colorés des pubs

In [ ]:
# ============================================================
# 2.1 — Chargement du modèle TrOCR
# ============================================================
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print('⏳ Chargement de TrOCR (microsoft/trocr-base-printed)...')
print('   Taille du modèle : ~334M paramètres')
print('   Pré-entraîné sur : millions de documents imprimés et manuscrits')

# Processor = Tokenizer + Feature Extractor combinés
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')

# Modèle = ViT Encoder + GPT-2 Decoder
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-printed')
model = model.to(device)
model.eval()  # Mode inférence

# Statistiques du modèle
total_params    = sum(p.numel() for p in model.parameters()) / 1e6
encoder_params  = sum(p.numel() for p in model.encoder.parameters()) / 1e6
decoder_params  = sum(p.numel() for p in model.decoder.parameters()) / 1e6

print(f'\n✅ TrOCR chargé sur {device} !')
print(f'   Paramètres totaux  : {total_params:.1f}M')
print(f'   Encoder (ViT)      : {encoder_params:.1f}M paramètres')
print(f'   Decoder (GPT-2)    : {decoder_params:.1f}M paramètres')

In [ ]:
# ============================================================
# 2.2 — Fonction d'inférence TrOCR
# ============================================================
def extract_text_trocr(image_path, return_attention=False):
    """
    Extrait le texte d'une image avec TrOCR.
    
    Étapes :
    1. Charge l'image en RGB
    2. Processor découpe en patches et normalise
    3. Encoder ViT extrait les features globales
    4. Decoder génère le texte token par token
    """
    debut = time.time()

    image = Image.open(image_path).convert('RGB')

    # Prétraitement : patches 16×16 + normalisation ImageNet
    pixel_values = processor(
        image,
        return_tensors='pt'
    ).pixel_values.to(device)

    # Génération du texte (beam search pour plus de précision)
    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            max_new_tokens=128,
            num_beams=4,          # Beam search : explore 4 séquences en parallèle
            output_attentions=return_attention,
            return_dict_in_generate=return_attention
        )

    # Décodage des IDs en texte lisible
    if return_attention:
        texte = processor.batch_decode(
            outputs.sequences, skip_special_tokens=True
        )[0]
        return texte, time.time() - debut, outputs, pixel_values
    else:
        texte = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        return texte, time.time() - debut


# Test rapide
img_test = os.path.join(IMAGES_TRAIN, TEST_IMAGES[0])
texte, duree = extract_text_trocr(img_test)
print(f'🧪 Test sur 1 image :')
print(f'   Texte extrait : "{texte}"')
print(f'   Durée         : {duree:.2f}s')
print('✅ Fonction définie !')

In [ ]:
# ============================================================
# 2.3 — Inférence sur les 20 images de test
# ============================================================
print('🔄 Application de TrOCR sur les 20 images de test...\n')

resultats   = []  # textes extraits
temps_liste = []  # temps par image
nb_mots     = []  # nombre de mots extraits
nb_chars    = []  # nombre de caractères

for i, img_name in enumerate(tqdm(TEST_IMAGES, desc='TrOCR')):
    img_path    = os.path.join(IMAGES_TRAIN, img_name)
    texte, duree = extract_text_trocr(img_path)
    resultats.append(texte)
    temps_liste.append(duree)
    nb_mots.append(len(texte.split()) if texte.strip() else 0)
    nb_chars.append(len(texte.strip()))

print(f'\n✅ TrOCR terminé !')
print(f'   Temps moyen par image : {np.mean(temps_liste):.2f}s')
print(f'   Temps total           : {sum(temps_liste):.2f}s')
print(f'   Images avec texte     : {sum(1 for t in resultats if t.strip())}/20')
print(f'   Mots extraits (moy)   : {np.mean(nb_mots):.1f} mots/image')
print('\n📋 Textes extraits :')
for i, (nom, texte) in enumerate(zip(TEST_IMAGES, resultats)):
    status = '✅' if texte.strip() else '❌'
    print(f'  {status} {i+1:2d}. "{texte[:70]}"' + ('...' if len(texte) > 70 else ''))

In [ ]:
# ============================================================
# 2.4 — Visualisation des résultats d'extraction
# ============================================================
fig, axes = plt.subplots(4, 5, figsize=(22, 18))
fig.suptitle('🔍 Résultats TrOCR — Texte extrait de chaque image',
             fontsize=14, fontweight='bold')

for idx, (ax, img_name, texte) in enumerate(zip(axes.flatten(), TEST_IMAGES, resultats)):
    img = Image.open(os.path.join(IMAGES_TRAIN, img_name)).convert('RGB')
    ax.imshow(img)
    ax.axis('off')

    # Afficher le texte extrait sous l'image
    texte_court = texte[:50] + '...' if len(texte) > 50 else texte
    if not texte.strip():
        texte_court = '(aucun texte)'
        color = '#e74c3c'
    else:
        color = '#27ae60'

    ax.set_title(f'#{idx+1}', fontsize=9, fontweight='bold')
    ax.text(0.5, -0.08, f'"{texte_court}"',
            transform=ax.transAxes, fontsize=7, ha='center', va='top',
            color=color, style='italic',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

plt.tight_layout()
plt.savefig('trocr_resultats.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Résultats sauvegardés')

---
# 🔬 PARTIE 3 — Explainability (XAI)

L'**Explainable AI (XAI)** nous permet de comprendre **pourquoi** le modèle prend ses décisions. C'est essentiel dans un projet forensique — on doit savoir sur quelles zones de l'image TrOCR se concentre.

Nous utilisons **3 méthodes** complémentaires :

| Méthode | Principe | Avantage |
|---------|----------|----------|
| **Attention Map** | Visualise les poids d'attention du Transformer | Natif au modèle, très précis |
| **Occlusion Sensitivity** | Cache des zones et mesure la dégradation | Intuitif, model-agnostic |
| **Gradient Saliency** | Gradient de la sortie par rapport à l'entrée | Rapide, pixel-level |

In [ ]:
# ============================================================
# 3.1 — XAI Méthode 1 : Attention Map (Cross-Attention)
# Le Transformer génère des poids d'attention qui montrent
# quels patches de l'image le decoder regarde pour générer
# chaque caractère du texte
# ============================================================
print('🔬 XAI Méthode 1 : Attention Map (Cross-Attention Decoder)')

# Sélectionner 3 images de test avec du texte détecté
imgs_avec_texte = [
    (img, txt) for img, txt in zip(TEST_IMAGES, resultats)
    if txt.strip()
][:3]

fig, axes = plt.subplots(len(imgs_avec_texte), 3, figsize=(18, 6 * len(imgs_avec_texte)))
fig.suptitle('🔬 XAI — Attention Map (quelles zones le modèle regarde)',
             fontsize=14, fontweight='bold')

if len(imgs_avec_texte) == 1:
    axes = [axes]

for row, (img_name, texte) in enumerate(imgs_avec_texte):
    img_path = os.path.join(IMAGES_TRAIN, img_name)
    img_pil  = Image.open(img_path).convert('RGB')

    # Extraire les attention maps
    pixel_values = processor(img_pil, return_tensors='pt').pixel_values.to(device)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values,
            max_new_tokens=64,
            output_attentions=True,
            return_dict_in_generate=True
        )

    # Récupérer la cross-attention du decoder vers l'encoder
    # cross_attentions[token][layer] → shape: (batch, heads, 1, num_patches)
    try:
        cross_attn = outputs.cross_attentions
        
        # cross_attn[0][-1][0] → shape: (heads, 1, num_patches)
        # num_patches = 577 = 576 patches + 1 token [CLS]
        # On enlève le premier token [CLS] avant de reshaper
        attn_map = cross_attn[0][-1][0].mean(0)[0].cpu().numpy()  # (577,)
        
        # Supprimer le token CLS (premier élément)
        attn_map = attn_map[1:]  # (576,) = 24×24 ✅
        
        # Remodeler en grille 24×24
        grid_size = int(attn_map.shape[0] ** 0.5)  # = 24
        attn_grid  = attn_map.reshape(grid_size, grid_size)
    
        # Upsampler vers la taille originale
        img_arr   = np.array(img_pil.resize((384, 384)))
        attn_up   = cv2.resize(attn_grid, (384, 384), interpolation=cv2.INTER_CUBIC)
        attn_norm = (attn_up - attn_up.min()) / (attn_up.max() - attn_up.min() + 1e-8)
    
        # Heatmap
        heatmap  = cv2.applyColorMap(np.uint8(255 * attn_norm), cv2.COLORMAP_JET)
        heatmap  = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay  = (0.55 * img_arr + 0.45 * heatmap).astype(np.uint8)
    
        # Colonne 1 : image originale
        axes[row][0].imshow(img_pil)
        axes[row][0].set_title('Image originale', fontweight='bold')
        axes[row][0].axis('off')
    
        # Colonne 2 : heatmap seule
        axes[row][1].imshow(attn_norm, cmap='hot')
        axes[row][1].set_title('Carte d\'attention', fontweight='bold')
        axes[row][1].axis('off')
    
        # Colonne 3 : superposition
        axes[row][2].imshow(overlay)
        axes[row][2].set_title(f'Superposition\n"{texte[:40]}"', fontweight='bold', fontsize=9)
        axes[row][2].axis('off')

    except Exception as e:
        print(f'   ⚠️ Attention map indisponible pour cette image : {e}')
        for col in range(3):
            axes[row][col].axis('off')

plt.tight_layout()
plt.savefig('xai_attention_map.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Attention Map sauvegardée — xai_attention_map.png')
print('💡 Rouge = zones très regardées | Bleu = zones ignorées')

In [ ]:
# ============================================================
# 3.2 — XAI Méthode 2 : Occlusion Sensitivity
# On cache des zones de l'image avec un carré noir,
# puis on mesure combien de texte est perdu.
# Si cacher une zone = perdre beaucoup de texte → zone importante
# ============================================================
print('🔬 XAI Méthode 2 : Occlusion Sensitivity')

def occlusion_sensitivity(img_path, grid_size=6, patch_size=64):
    """
    Divise l'image en grid_size×grid_size régions.
    Pour chaque région, cache-la avec un carré gris et
    mesure combien de caractères sont perdus.
    Retourne une heatmap d'importance.
    """
    img_orig = Image.open(img_path).convert('RGB')
    img_arr  = np.array(img_orig)
    H, W     = img_arr.shape[:2]

    # Référence : texte complet sans occlusion
    ref_text, _ = extract_text_trocr(img_path)
    ref_len      = len(ref_text)

    importance = np.zeros((grid_size, grid_size))
    step_h = H // grid_size
    step_w = W // grid_size

    for i in range(grid_size):
        for j in range(grid_size):
            # Créer une copie avec la zone (i,j) masquée
            img_masked        = img_arr.copy()
            y1, y2            = i * step_h, min((i+1) * step_h, H)
            x1, x2            = j * step_w, min((j+1) * step_w, W)
            img_masked[y1:y2, x1:x2] = 128  # Gris neutre

            img_masked_pil = Image.fromarray(img_masked)
            pixel_values   = processor(img_masked_pil, return_tensors='pt').pixel_values.to(device)

            with torch.no_grad():
                gen  = model.generate(pixel_values, max_new_tokens=64)
                text = processor.batch_decode(gen, skip_special_tokens=True)[0]

            # Score d'importance = perte de caractères
            importance[i, j] = max(0, ref_len - len(text))

    return importance, ref_text, img_orig


# Appliquer sur 2 images avec texte détecté
test_imgs_occ = [(img, txt) for img, txt in zip(TEST_IMAGES, resultats) if txt.strip()][:2]

fig, axes = plt.subplots(len(test_imgs_occ), 3, figsize=(18, 6 * len(test_imgs_occ)))
fig.suptitle('🔬 XAI — Occlusion Sensitivity (zones critiques pour l\'OCR)',
             fontsize=14, fontweight='bold')

if len(test_imgs_occ) == 1:
    axes = [axes]

for row, (img_name, texte) in enumerate(test_imgs_occ):
    img_path = os.path.join(IMAGES_TRAIN, img_name)
    print(f'  Calcul occlusion pour image {row+1}/{len(test_imgs_occ)}...')

    importance, ref_text, img_pil = occlusion_sensitivity(img_path, grid_size=6)

    # Normaliser
    imp_norm = (importance - importance.min()) / (importance.max() - importance.min() + 1e-8)

    # Upsampler
    img_arr  = np.array(img_pil.resize((384, 384)))
    imp_up   = cv2.resize(imp_norm, (384, 384), interpolation=cv2.INTER_NEAREST)
    heatmap  = cv2.applyColorMap(np.uint8(255 * imp_up), cv2.COLORMAP_INFERNO)
    heatmap  = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay  = (0.55 * img_arr + 0.45 * heatmap).astype(np.uint8)

    axes[row][0].imshow(img_pil)
    axes[row][0].set_title('Image originale', fontweight='bold')
    axes[row][0].axis('off')

    im = axes[row][1].imshow(importance, cmap='inferno', interpolation='nearest')
    axes[row][1].set_title('Grille d\'importance (6×6)', fontweight='bold')
    axes[row][1].axis('off')
    plt.colorbar(im, ax=axes[row][1], fraction=0.046, pad=0.04)

    axes[row][2].imshow(overlay)
    axes[row][2].set_title(f'Superposition\n"{ref_text[:40]}"', fontweight='bold', fontsize=9)
    axes[row][2].axis('off')

plt.tight_layout()
plt.savefig('xai_occlusion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Occlusion Sensitivity sauvegardée — xai_occlusion.png')
print('💡 Jaune/blanc = zones critiques | Noir = zones sans impact')

In [ ]:
# ============================================================
# 3.3 — XAI Méthode 3 : Gradient Saliency
# Calcule le gradient de la première sortie générée
# par rapport aux pixel values d'entrée.
# Les pixels avec les plus grands gradients sont les
# plus influents pour la décision du modèle.
# ============================================================
print('🔬 XAI Méthode 3 : Gradient Saliency Map')

def gradient_saliency(img_path):
    """
    Calcule la saliency map par gradient :
    - Forward pass pour obtenir les logits du premier token
    - Backward pass pour calculer d/d(pixels) du logit le plus probable
    - Valeur absolue des gradients = importance de chaque pixel
    """
    img_pil      = Image.open(img_path).convert('RGB')
    pixel_values = processor(img_pil, return_tensors='pt').pixel_values.to(device)
    pixel_values.requires_grad_(True)  # Activer le calcul de gradient

    # Décoder les ids du début (token de début BOS)
    # Pour VisionEncoderDecoderModel, le token BOS est dans decoder.config
    bos_token_id = model.decoder.config.decoder_start_token_id or model.decoder.config.bos_token_id
    decoder_input_ids = torch.tensor([[bos_token_id]]).to(device)

    # Forward pass
    outputs = model(
        pixel_values=pixel_values,
        decoder_input_ids=decoder_input_ids
    )

    # Logits du premier token prédit
    logits       = outputs.logits[0, 0]  # (vocab_size,)
    top_token    = logits.argmax().item()
    score        = logits[top_token]

    # Backward
    model.zero_grad()
    score.backward()

    # Gradient par rapport aux pixels
    saliency = pixel_values.grad.data.abs()  # (1, 3, H, W)
    saliency = saliency.squeeze().mean(0)    # (H, W) — moyenne sur les 3 canaux
    saliency = saliency.cpu().numpy()
    saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)

    return saliency, img_pil


# Appliquer sur 3 images avec texte
test_imgs_sal = [(img, txt) for img, txt in zip(TEST_IMAGES, resultats) if txt.strip()][:3]

fig, axes = plt.subplots(len(test_imgs_sal), 3, figsize=(18, 6 * len(test_imgs_sal)))
fig.suptitle('🔬 XAI — Gradient Saliency (pixels les plus influents)',
             fontsize=14, fontweight='bold')

if len(test_imgs_sal) == 1:
    axes = [axes]

for row, (img_name, texte) in enumerate(test_imgs_sal):
    img_path = os.path.join(IMAGES_TRAIN, img_name)

    saliency, img_pil = gradient_saliency(img_path)

    # Upsampler saliency à la taille originale
    img_arr    = np.array(img_pil)
    H, W       = img_arr.shape[:2]
    sal_up     = cv2.resize(saliency, (W, H), interpolation=cv2.INTER_CUBIC)
    heatmap    = cv2.applyColorMap(np.uint8(255 * sal_up), cv2.COLORMAP_VIRIDIS)
    heatmap    = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    overlay    = (0.6 * img_arr + 0.4 * heatmap).astype(np.uint8)

    axes[row][0].imshow(img_pil)
    axes[row][0].set_title('Image originale', fontweight='bold')
    axes[row][0].axis('off')

    axes[row][1].imshow(sal_up, cmap='viridis')
    axes[row][1].set_title('Gradient Saliency', fontweight='bold')
    axes[row][1].axis('off')

    axes[row][2].imshow(overlay)
    axes[row][2].set_title(f'Superposition\n"{texte[:40]}"', fontweight='bold', fontsize=9)
    axes[row][2].axis('off')

plt.tight_layout()
plt.savefig('xai_saliency.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gradient Saliency sauvegardée — xai_saliency.png')
print('💡 Jaune/clair = pixels très importants | Violet/noir = pixels ignorés')

---
# 📈 PARTIE 4 — Évaluation des Performances

On évalue TrOCR avec **plusieurs métriques** couvrant différentes dimensions de la qualité OCR.

| Métrique | Ce qu'elle mesure | Idéalement |
|----------|-------------------|------------|
| **Taux de détection** | % images avec texte trouvé | ↑ élevé |
| **CER** | Erreurs au niveau caractère | ↓ faible |
| **WER** | Erreurs au niveau mot | ↓ faible |
| **Temps inférence** | Vitesse de traitement | ↓ faible |
| **Nb mots extraits** | Richesse de l'extraction | ↑ élevé |
| **Confiance** | Certitude du modèle | ↑ élevée |

In [ ]:
# ============================================================
# 4.1 — Calcul des métriques CER et WER
# ============================================================
def cer(ref, hyp):
    """Character Error Rate = distance édition / longueur référence"""
    r, h = list(ref.lower()), list(hyp.lower())
    if not r: return 0.0 if not h else 1.0
    d = np.zeros((len(r)+1, len(h)+1))
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = d[i-1][j-1] if r[i-1]==h[j-1] else 1+min(d[i-1][j], d[i][j-1], d[i-1][j-1])
    return d[len(r)][len(h)] / len(r)

def wer(ref, hyp):
    """Word Error Rate = distance édition / nombre de mots référence"""
    r, h = ref.lower().split(), hyp.lower().split()
    if not r: return 0.0 if not h else 1.0
    d = np.zeros((len(r)+1, len(h)+1))
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            d[i][j] = d[i-1][j-1] if r[i-1]==h[j-1] else 1+min(d[i-1][j], d[i][j-1], d[i-1][j-1])
    return d[len(r)][len(h)] / len(r)


# Utiliser le texte extrait sur l'image 1 comme référence
# (sans ground truth annoté, on compare les images entre elles)
reference_texte = resultats[0] if resultats[0].strip() else 'text sample'

# Calculer CER et WER pour chaque image vs la référence
cer_scores = [cer(reference_texte, t) if t.strip() else 1.0 for t in resultats]
wer_scores = [wer(reference_texte, t) if t.strip() else 1.0 for t in resultats]

# Statistiques globales
detection_rate = sum(1 for t in resultats if t.strip()) / len(resultats) * 100
avg_mots       = np.mean([len(t.split()) for t in resultats if t.strip()]) if any(t.strip() for t in resultats) else 0
avg_chars      = np.mean([len(t) for t in resultats if t.strip()]) if any(t.strip() for t in resultats) else 0

print('='*60)
print('📊 MÉTRIQUES DE PERFORMANCE — TrOCR sur COCO 2014')
print('='*60)
print(f'  Taux de détection     : {detection_rate:.1f}%  ({sum(1 for t in resultats if t.strip())}/20 images)')
print(f'  Temps moyen/image     : {np.mean(temps_liste):.3f}s')
print(f'  Temps total (20 imgs) : {sum(temps_liste):.2f}s')
print(f'  Mots extraits (moy)   : {avg_mots:.1f} mots/image')
print(f'  Chars extraits (moy)  : {avg_chars:.1f} chars/image')
print(f'  CER moyen             : {np.mean(cer_scores):.4f} ({np.mean(cer_scores)*100:.1f}% d\'erreur)')
print(f'  WER moyen             : {np.mean(wer_scores):.4f} ({np.mean(wer_scores)*100:.1f}% d\'erreur)')
print('='*60)

In [ ]:
# ============================================================
# 4.2 — Tableau de résultats par image
# ============================================================
df_results = pd.DataFrame({
    'Image'     : [f'Test #{i+1}' for i in range(len(TEST_IMAGES))],
    'Texte extrait' : [t[:50]+'...' if len(t)>50 else t for t in resultats],
    'Nb mots'   : nb_mots,
    'Nb chars'  : nb_chars,
    'Temps (s)' : [round(t, 3) for t in temps_liste],
    'Détecté'   : ['✅' if t.strip() else '❌' for t in resultats],
})

print('📋 TABLEAU DÉTAILLÉ PAR IMAGE')
print(df_results.to_string(index=False))

In [ ]:
# ============================================================
# 4.3 — Visualisations des métriques (6 graphiques)
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('📊 Analyse complète des performances TrOCR',
             fontsize=16, fontweight='bold')

imgs_ids = [f'#{i+1}' for i in range(len(TEST_IMAGES))]

# ── Graphique 1 : Taux de détection ──────────────────────────
detected  = sum(1 for t in resultats if t.strip())
sizes_pie = [detected, len(resultats) - detected]
colors    = ['#27ae60', '#e74c3c']
explode   = (0.05, 0)
wedges, texts, autotexts = axes[0][0].pie(
    sizes_pie, labels=['Texte détecté', 'Aucun texte'],
    colors=colors, autopct='%1.1f%%', explode=explode,
    startangle=90, textprops={'fontsize': 11}
)
for at in autotexts: at.set_fontweight('bold')
axes[0][0].set_title('Taux de détection', fontweight='bold', fontsize=12)

# ── Graphique 2 : Temps d'inférence par image ─────────────────
bar_colors = ['#3498db' if t.strip() else '#e74c3c' for t in resultats]
axes[0][1].bar(imgs_ids, temps_liste, color=bar_colors, edgecolor='black', linewidth=0.5)
axes[0][1].axhline(np.mean(temps_liste), color='orange', linestyle='--',
                    linewidth=2, label=f'Moy: {np.mean(temps_liste):.2f}s')
axes[0][1].set_title('Temps d\'inférence par image', fontweight='bold', fontsize=12)
axes[0][1].set_xlabel('Image')
axes[0][1].set_ylabel('Secondes')
axes[0][1].tick_params(axis='x', rotation=45)
axes[0][1].legend()
blue_p  = mpatches.Patch(color='#3498db', label='Texte détecté')
red_p   = mpatches.Patch(color='#e74c3c', label='Aucun texte')
axes[0][1].legend(handles=[blue_p, red_p, axes[0][1].get_lines()[0]])

# ── Graphique 3 : Nombre de mots extraits ─────────────────────
axes[0][2].bar(imgs_ids, nb_mots, color='#9b59b6', edgecolor='black', linewidth=0.5)
axes[0][2].axhline(np.mean(nb_mots), color='red', linestyle='--',
                    linewidth=2, label=f'Moy: {np.mean(nb_mots):.1f}')
axes[0][2].set_title('Nombre de mots extraits', fontweight='bold', fontsize=12)
axes[0][2].set_xlabel('Image')
axes[0][2].set_ylabel('Mots')
axes[0][2].tick_params(axis='x', rotation=45)
axes[0][2].legend()

# ── Graphique 4 : Distribution du nombre de mots ─────────────
mots_nonzero = [m for m in nb_mots if m > 0]
if mots_nonzero:
    axes[1][0].hist(mots_nonzero, bins=10, color='#e67e22', edgecolor='black')
    axes[1][0].axvline(np.mean(mots_nonzero), color='red', linestyle='--',
                        label=f'Moy: {np.mean(mots_nonzero):.1f}')
    axes[1][0].set_title('Distribution des mots extraits\n(images avec texte seulement)',
                          fontweight='bold', fontsize=11)
    axes[1][0].set_xlabel('Nombre de mots')
    axes[1][0].set_ylabel('Fréquence')
    axes[1][0].legend()

# ── Graphique 5 : CER par image ───────────────────────────────
cer_colors = ['#27ae60' if c < 0.3 else '#f39c12' if c < 0.7 else '#e74c3c' for c in cer_scores]
axes[1][1].bar(imgs_ids, cer_scores, color=cer_colors, edgecolor='black', linewidth=0.5)
axes[1][1].axhline(np.mean(cer_scores), color='black', linestyle='--',
                    linewidth=2, label=f'CER moy: {np.mean(cer_scores):.3f}')
axes[1][1].set_title('CER par image\n(↓ = meilleur)', fontweight='bold', fontsize=12)
axes[1][1].set_xlabel('Image')
axes[1][1].set_ylabel('CER (0=parfait, 1=tout faux)')
axes[1][1].set_ylim(0, 1.2)
axes[1][1].tick_params(axis='x', rotation=45)
axes[1][1].legend()
g_p = mpatches.Patch(color='#27ae60', label='CER < 0.3 (bon)')
o_p = mpatches.Patch(color='#f39c12', label='CER 0.3-0.7 (moyen)')
r_p = mpatches.Patch(color='#e74c3c', label='CER > 0.7 (mauvais)')
axes[1][1].legend(handles=[g_p, o_p, r_p, axes[1][1].get_lines()[0]], fontsize=8)

# ── Graphique 6 : Synthèse radar des métriques clés ───────────
categories    = ['Détection\n(%)', 'Vitesse\n(1/s×10)', 'Mots\nextraits', 'CER\n(inv×100)']
speed_score   = (1 / np.mean(temps_liste)) * 10
values_radar  = [
    detection_rate,
    min(speed_score, 100),
    min(avg_mots * 10, 100),
    max(0, (1 - np.mean(cer_scores)) * 100)
]
bars6 = axes[1][2].bar(categories, values_radar,
                        color=['#27ae60', '#3498db', '#9b59b6', '#e67e22'],
                        edgecolor='black', linewidth=0.8)
axes[1][2].set_title('Synthèse des métriques TrOCR\n(scores normalisés)',
                      fontweight='bold', fontsize=12)
axes[1][2].set_ylim(0, 120)
axes[1][2].set_ylabel('Score (plus élevé = meilleur)')
for bar, val in zip(bars6, values_radar):
    axes[1][2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                   f'{val:.1f}', ha='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('performance_complete.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Analyse des performances sauvegardée — performance_complete.png')

In [ ]:
# ============================================================
# 4.4 — Analyse de la longueur des textes extraits
# et fréquence des mots les plus courants
# ============================================================
tous_mots = []
for texte in resultats:
    if texte.strip():
        mots = texte.lower().split()
        tous_mots.extend(mots)

if tous_mots:
    freq = Counter(tous_mots).most_common(15)
    mots_top, counts_top = zip(*freq)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('🔤 Analyse linguistique du texte extrait par TrOCR',
                 fontsize=13, fontweight='bold')

    # Top 15 mots
    colors_bar = plt.cm.viridis(np.linspace(0.2, 0.8, len(mots_top)))
    axes[0].barh(list(mots_top)[::-1], list(counts_top)[::-1],
                 color=colors_bar[::-1], edgecolor='black', linewidth=0.5)
    axes[0].set_title('Top 15 mots les plus fréquents', fontweight='bold')
    axes[0].set_xlabel('Fréquence')

    # Distribution longueur des textes
    longueurs = [len(t) for t in resultats if t.strip()]
    axes[1].hist(longueurs, bins=10, color='#1abc9c', edgecolor='black')
    axes[1].axvline(np.mean(longueurs), color='red', linestyle='--',
                    label=f'Moy: {np.mean(longueurs):.0f} chars')
    axes[1].set_title('Distribution longueur des textes\n(en caractères)', fontweight='bold')
    axes[1].set_xlabel('Longueur (caractères)')
    axes[1].set_ylabel('Fréquence')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('analyse_linguistique.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✅ Analyse linguistique sauvegardée')
else:
    print('⚠️ Pas assez de texte extrait pour l\'analyse linguistique')

In [ ]:
# ============================================================
# 4.5 — Résumé final et conclusion
# ============================================================
print('='*65)
print('🏆 RÉSUMÉ FINAL — TrOCR sur COCO 2014')
print('='*65)
print(f'\n  📊 Dataset :')
print(f'     Images totales   : {len(all_images):,}')
print(f'     Images testées   : {len(TEST_IMAGES)}')
print(f'\n  🤖 Modèle :')
print(f'     Architecture     : ViT Encoder + GPT-2 Decoder')
print(f'     Paramètres       : ~334M')
print(f'     Pré-entraînement : IIT-CDIP + SROIE + IAM')
print(f'\n  📈 Performances :')
print(f'     Taux de détection : {detection_rate:.1f}%')
print(f'     Temps moyen       : {np.mean(temps_liste):.3f}s/image')
print(f'     CER moyen         : {np.mean(cer_scores):.4f}')
print(f'     WER moyen         : {np.mean(wer_scores):.4f}')
print(f'     Mots/image (moy)  : {avg_mots:.1f}')
print(f'\n  🔬 XAI — 3 méthodes appliquées :')
print(f'     1. Attention Map    → zones regardées par le Transformer')
print(f'     2. Occlusion Sensitivity → zones critiques pour l\'OCR')
print(f'     3. Gradient Saliency → pixels les plus influents')
print(f'\n  🔜 Prochaine étape :')
print(f'     → SO1-B : classifier le texte extrait avec RoBERTa')
print(f'       (manipulateur vs neutre)')
print('='*65)
print('✅ SO1-A TERMINÉ — Notebook prêt pour la validation !')
print('='*65)

In [ ]:
# ============================================================
# 4.6 — Liste de tous les fichiers générés
# ============================================================
import glob

fichiers = glob.glob('/kaggle/working/*.png') + glob.glob('/kaggle/working/*.jpg')
print('📁 FICHIERS GÉNÉRÉS PAR CE NOTEBOOK :')
for f in sorted(fichiers):
    taille = os.path.getsize(f) / 1024
    print(f'   📊 {os.path.basename(f):40s} ({taille:.1f} KB)')